In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# LOAD DATA
train = pd.read_excel("case_1_labeled_data.xlsx")

# ENCODE LABEL
label_encoder = LabelEncoder()
train['label_encoded'] = label_encoder.fit_transform(train['label'])

# SPLIT DATA
X_train, X_val, y_train, y_val = train_test_split(
    train['full_text'],
    train['label_encoded'],
    test_size=0.2,
    stratify=train['label_encoded'],
    random_state=42
)

# HUGGINGFACE DATASET
train_dataset = Dataset.from_dict({
    'text': X_train.tolist(),
    'label': y_train.tolist()
})

val_dataset = Dataset.from_dict({
    'text': X_val.tolist(),
    'label': y_val.tolist()
})

# MODEL NAME
model_name = "indobenchmark/indobert-base-p1"

# TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(model_name)

# TOKENIZATION
def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

# MODEL
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

# METRIC
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    score = balanced_accuracy_score(labels, predictions)

    return {
        'balanced_accuracy': score
    }

# TRAINING CONFIG
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50
)

# TRAINER
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# TRAIN
trainer.train()

# EVALUATE
results = trainer.evaluate()

print(results)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

C:\Users\rifky\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rifky\.cache\huggingface\hub\models--indobenchmark--indobert-base-p1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

C:\Users\rifky\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\rifky\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Balanced Accuracy
0.703495,1.048300,2,0.652207


{'eval_loss': 1.0482999086380005, 'eval_balanced_accuracy': 0.6522070513572373}


In [2]:
# LOAD TEST DATA
test = pd.read_excel("case_1_text_to_predict.xlsx")

# CONVERT TO DATASET
test_dataset = Dataset.from_dict({
    'text': test['full_text'].tolist()
})

# TOKENIZE TEST DATA
test_dataset = test_dataset.map(tokenize, batched=True)

# PREDICT
predictions = trainer.predict(test_dataset)

# AMBIL LABEL PREDIKSI
pred_labels = np.argmax(predictions.predictions, axis=-1)

# CONVERT ANGKA -> LABEL ASLI
predicted_labels = label_encoder.inverse_transform(pred_labels)

# LOAD TEMPLATE
submission = pd.read_excel("case_1_template_sheet.xlsx")

# ISI LABEL
submission['label'] = predicted_labels

# SAVE SUBMISSION
submission.to_excel("submission_indobert.xlsx", index=False)

print(submission.head())
print("Submission berhasil dibuat!")

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

C:\Users\rifky\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


        id             label
0  TXT0001        Distribusi
1  TXT0002           Lainnya
2  TXT0003   Kualitas Pangan
3  TXT0004  Sasaran Penerima
4  TXT0005   Kualitas Pangan
Submission berhasil dibuat!
